# Week 5 -- Function 6: GP + EI + SVM Active Filter (5D)

In [1]:
# -- WEEKLY OBSERVATIONS LOG
import numpy as np

INITIAL_SIZE = 20
DATA_PATH_X  = '../data/function_6/initial_inputs.npy'
DATA_PATH_Y  = '../data/function_6/initial_outputs.npy'

weekly_log = [
    ([0.705342, 0.105077, 0.763664, 0.783759, 0.051342], -0.695846),  # W1: -0.696
    ([0.770275, 0.188487, 0.822811, 0.73851, 0.068343], -0.755348),  # W2: slightly worse
    ([0.606716, 0.166787, 0.797421, 0.720111, 0.086136], -0.557288),  # W3: BEST
    ([0.406716, 0.005417, 0.997421, 0.920111, 0.0], -0.981732150456315),  # W4: -0.982 (worst)
    # Week 5: add result here as ([x...], y_value)
]

X_disk = np.load(DATA_PATH_X)
Y_disk = np.load(DATA_PATH_Y)

n_missing = (INITIAL_SIZE + len(weekly_log)) - len(Y_disk)
if n_missing > 0:
    new_entries = weekly_log[len(weekly_log) - n_missing:]
    for x_vals, y_val in new_entries:
        X_disk = np.vstack([X_disk, np.array([x_vals])])
        Y_disk = np.append(Y_disk, y_val)
    np.save(DATA_PATH_X, X_disk)
    np.save(DATA_PATH_Y, Y_disk)
    print(f'Synced {n_missing} new observation(s).')
else:
    print('Already up-to-date.')

print(f'X: {X_disk.shape} | Y: {Y_disk.shape}')
current_week = len(Y_disk) - INITIAL_SIZE + 1
print(f'Week {current_week} | {len(Y_disk)} total observations ({INITIAL_SIZE} initial + {len(Y_disk)-INITIAL_SIZE} added)')

Synced 1 new observation(s).
X: (24, 5) | Y: (24,)
Week 5 | 24 total observations (20 initial + 4 added)


In [2]:
# Cell 3: Load data and inspect -- Function 6: Environmental Monitoring (5D), Maximise

X = np.load(DATA_PATH_X)
Y = np.load(DATA_PATH_Y)
n_dims = X.shape[1]

print(f'Input  shape : {X.shape}')
print(f'Output shape : {Y.shape}')
print()

pairs = sorted(zip(Y.tolist(), X.tolist()), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 72)
print('  All observations (sorted descending by Y)')
print('=' * 72)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    x_str = ', '.join(f'{v:.6f}' for v in x_val)
    print(f'  [{i+1:2d}]  X = [{x_str}]   Y = {y_val:+.6e}{marker}')
print('=' * 72)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
best_x_str = ', '.join(f'{v:.8f}' for v in best_X)
print(f'\n  Best Y*  = {best_Y:.6e}')
print(f'  Best X*  = [{best_x_str}]')

Input  shape : (24, 5)
Output shape : (24,)

  All observations (sorted descending by Y)
  [ 1]  X = [0.606716, 0.166787, 0.797421, 0.720111, 0.086136]   Y = -5.572880e-01  <-- best
  [ 2]  X = [0.705342, 0.105077, 0.763664, 0.783759, 0.051342]   Y = -6.958464e-01
  [ 3]  X = [0.728186, 0.154693, 0.732552, 0.693997, 0.056401]   Y = -7.142649e-01
  [ 4]  X = [0.770275, 0.188487, 0.822811, 0.738510, 0.068343]   Y = -7.553483e-01
  [ 5]  X = [0.618812, 0.331802, 0.187288, 0.756238, 0.328835]   Y = -8.292366e-01
  [ 6]  X = [0.782880, 0.536336, 0.443284, 0.859700, 0.010326]   Y = -9.357566e-01
  [ 7]  X = [0.406716, 0.005417, 0.997421, 0.920111, 0.000000]   Y = -9.817322e-01
  [ 8]  X = [0.536797, 0.308781, 0.411879, 0.388225, 0.522528]   Y = -1.144785e+00
  [ 9]  X = [0.242384, 0.844100, 0.577809, 0.679021, 0.501953]   Y = -1.209955e+00
  [10]  X = [0.145111, 0.896685, 0.896322, 0.726272, 0.236272]   Y = -1.233786e+00
  [11]  X = [0.784958, 0.910682, 0.708120, 0.959225, 0.004911]   Y = -1

In [3]:
# Cell 4: Fit GP
import warnings
warnings.filterwarnings('ignore')
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF

Y_fit = np.log(np.abs(Y) + 1e-300)
kernel = RBF(length_scale=0.1, length_scale_bounds='fixed')
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10)
gp.fit(X, Y_fit)

print('=' * 62)
print('  GP Fitting Results')
print('=' * 62)
print(f'  Kernel                  : {gp.kernel_}')
print(f'  Log-marginal-likelihood : {gp.log_marginal_likelihood_value_:.4f}')
pred_mean, pred_std = gp.predict(best_X.reshape(1, -1), return_std=True)
actual_log = np.log(np.abs(best_Y) + 1e-300)
print(f'  Sanity check at best X* : GP mean={pred_mean[0]:.4f}  actual={actual_log:.4f}  std={pred_std[0]:.6f}')
print('=' * 62)

  GP Fitting Results
  Kernel                  : RBF(length_scale=0.1)
  Log-marginal-likelihood : -23.9799
  Sanity check at best X* : GP mean=-0.5847  actual=-0.5847  std=0.000010


In [4]:
# Cell 5: GP EI Acquisition (GP-only -- no NN)
from scipy.stats import norm

best_log_y = np.log(np.abs(best_Y) + 1e-300)
np.random.seed(42)
X_grid = np.random.uniform(0, 1, size=(30000, n_dims))
gp_mean, gp_std = gp.predict(X_grid, return_std=True)
improvement = gp_mean - best_log_y
Z = improvement / (gp_std + 1e-9)
ei_scores = improvement * norm.cdf(Z) + gp_std * norm.pdf(Z)
gp_candidate = X_grid[np.argmax(ei_scores)]

# Preliminary selection (SVM may override in next cell)
next_x = gp_candidate.copy()
portal_string = '-'.join([f'{v:.6f}' for v in next_x])
selected = 'GP EI'

print('=' * 62)
print('  GP EI Acquisition')
print('=' * 62)
gp_str = ', '.join(f'{v:.6f}' for v in gp_candidate)
print(f'  GP EI candidate : [{gp_str}]')
print(f'  (Preliminary -- SVM filter applied in next cell)')
print('=' * 62)

  GP EI Acquisition
  GP EI candidate : [0.979258, 0.261953, 0.097640, 0.160574, 0.019166]
  (Preliminary -- SVM filter applied in next cell)


In [5]:
# Cell 6: SVM Active Filter
# Checks GP EI candidate. If flagged poor, generates a backup near best_X.
from sklearn.svm import SVC

threshold = np.percentile(Y, 70)
labels = (Y >= threshold).astype(int)

print(f'  SVM threshold (70th pct): {threshold:.6e}')
print(f'  Good samples: {labels.sum()} / {len(labels)}')
print()

if labels.sum() >= 2 and (labels == 0).sum() >= 2:
    svm_clf = SVC(kernel='rbf', C=1.0, gamma='scale', probability=True, random_state=42)
    svm_clf.fit(X, labels)

    prob_gp = svm_clf.predict_proba(gp_candidate.reshape(1, -1))[0, 1]
    print(f'  P(good) -- GP EI candidate: {prob_gp:.3f}')
    print()

    if prob_gp >= 0.3:
        svm_selected = 'GP EI candidate (SVM endorsed)'
        print(f'  SVM endorses GP EI candidate.')
    else:
        np.random.seed(99)
        backup = np.clip(
            best_X + np.random.uniform(-0.15, 0.15, size=best_X.shape),
            0.0, 1.0)
        prob_backup = svm_clf.predict_proba(backup.reshape(1, -1))[0, 1]
        backup_str = ', '.join(f'{v:.6f}' for v in backup)
        print(f'  GP EI candidate flagged poor (P={prob_gp:.3f}).')
        print(f'  Backup near best_X: [{backup_str}]  P(good)={prob_backup:.3f}')
        if prob_backup > prob_gp:
            next_x = backup.copy()
            svm_selected = 'Backup near best_X (SVM preferred over GP EI)'
            portal_string = '-'.join([f'{v:.6f}' for v in next_x])
            print('  SVM selects backup candidate.')
        else:
            svm_selected = 'GP EI candidate (SVM fallback -- backup also poor)'
            print('  Both poor. Keeping GP EI candidate.')
else:
    svm_selected = 'GP EI candidate (insufficient SVM data)'
    print('  Insufficient class balance -- keeping GP EI candidate.')

next_str = ', '.join(f'{v:.6f}' for v in next_x)
print(f'\n  FINAL query : [{next_str}]')
print(f'  Portal      : >>> {portal_string} <<<')

  SVM threshold (70th pct): -1.128480e+00
  Good samples: 7 / 24

  P(good) -- GP EI candidate: 0.059

  GP EI candidate flagged poor (P=0.059).
  Backup near best_X: [0.658400, 0.163211, 0.895070, 0.579545, 0.178551]  P(good)=0.810
  SVM selects backup candidate.

  FINAL query : [0.658400, 0.163211, 0.895070, 0.579545, 0.178551]
  Portal      : >>> 0.658400-0.163211-0.895070-0.579545-0.178551 <<<


In [6]:
print('=' * 72)
print('  SUMMARY -- Week 5 Bayesian Optimisation')
print('=' * 72)
print('  Function        : 6 -- Environmental Monitoring (5D)')
print('  Objective       : Maximise')
print(f'  Method          : GP EI + SVM active filter')
print(f'  SVM decision    : {svm_selected}')
print()
best_x_s = ', '.join(f'{v:.6f}' for v in best_X)
next_s   = ', '.join(f'{v:.6f}' for v in next_x)
print(f'  Best X*         : [{best_x_s}]')
print(f'  Best Y*         : {best_Y:.6e}')
print()
print(f'  Next query      : [{next_s}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 72)

  SUMMARY -- Week 5 Bayesian Optimisation
  Function        : 6 -- Environmental Monitoring (5D)
  Objective       : Maximise
  Method          : GP EI + SVM active filter
  SVM decision    : Backup near best_X (SVM preferred over GP EI)

  Best X*         : [0.606716, 0.166787, 0.797421, 0.720111, 0.086136]
  Best Y*         : -5.572880e-01

  Next query      : [0.658400, 0.163211, 0.895070, 0.579545, 0.178551]

  Portal submission string:
  >>> 0.658400-0.163211-0.895070-0.579545-0.178551 <<<


## Week 5 -- Reflection

**Function**: F6 -- Environmental Monitoring (5D)

### Strategy note
W4 worst ever. GP+EI from W3 best. SVM active filter.

### Query
- **Input submitted**: *(fill in portal submission string)*
- **Output received**: *(fill in after result)*
- **Best so far**: *(update after result)*

### What this result taught us
*(fill in after receiving result)*

### Strategy for Week 6
*(fill in after reflecting on result)*